## __Baseline Models — Random Forest & Logistic Regression__

Working patient's data through RFC and LR models to check whether our more complex model is actually justified for this analysis.

### __Imports__

In [ ]:
import pandas as pd, os
import numpy as np
 
from sklearn.preprocessing import StandardScaler 
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from dotenv import load_dotenv

load_dotenv("project.env")

True

### __Random Forest Classifier__

### Import dataset

In [ ]:
X_clinical_test = pd.read_csv(os.path.join(os.getenv("Output_path"), "X_clinical_test.csv")).set_index('file_id')
X_clinical_train = pd.read_csv(os.path.join(os.getenv("Output_path"), "X_clinical_train.csv")).set_index('file_id')

X_expression_test = pd.read_csv(os.path.join(os.getenv("Output_path"), "X_expression_raw_test.csv")).set_index('file_id')
X_expression_train = pd.read_csv(os.path.join(os.getenv("Output_path"), "X_expression_raw_train.csv")).set_index('file_id')

X_expression_D2_test = pd.read_csv(os.path.join(os.getenv("Output_path"), "X_expression_D2_test.csv")).set_index('file_id')
X_expression_D2_train = pd.read_csv(os.path.join(os.getenv("Output_path"), "X_expression_D2_train.csv")).set_index('file_id')

y_test = pd.read_csv(os.path.join(os.getenv("Output_path"), "y_test.csv")).set_index('file_id')
y_train = pd.read_csv(os.path.join(os.getenv("Output_path"), "y_train.csv")).set_index('file_id')

In [3]:
scaler = StandardScaler()

### Creating datasets

#### PCA

In [4]:
df_train_ = pd.concat([X_expression_train, X_clinical_train], axis=1)
df_test_ = pd.concat([X_expression_test, X_clinical_test], axis=1)

In [5]:
X_train_ = scaler.fit_transform(df_train_)
X_test_ = scaler.transform(df_test_) 

pca = PCA(n_components=50) ## Save 95% of the dimensionality within the variables of the expression dataset
X_train_PCA = pca.fit_transform(X_train_)
X_test_PCA = pca.transform(X_test_)

In [ ]:
cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'n_estimators': [50, 100, 150],  
    'max_depth': [None, 10, 20], 
    'min_samples_split': [2, 5, 10],  
    'min_samples_leaf': [1, 2, 4], 
    'max_features': ['sqrt', 'log2']
}

classifier = RandomForestClassifier(class_weight='balanced')

Forest_grid_PCA = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=param_grid,   
    n_iter=20,                        
    cv=cross_val,
    n_jobs=1,                         
    scoring='recall',
    random_state=42, 
    verbose = 3
)


Forest_grid_PCA.fit(X_train_PCA, y_train.values.ravel())
y_pred_single_PCA = Forest_grid_PCA.predict(X_test_PCA)
accuracy = accuracy_score(y_test, y_pred_single_PCA)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(confusion_matrix(y_test.values.ravel(), y_pred_single_PCA))
pd.DataFrame(confusion_matrix(y_test.values.ravel(), y_pred_single_PCA)).to_excel(os.path.join(os.getenv("Output_path"), 'confusion_matrix_RandomForest_PCA.xlsx'), sheet_name= 'sheet1', header = True)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.919 total time=   0.0s
[CV 2/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.946 total time=   0.0s
[CV 3/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.784 total time=   0.0s
[CV 4/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.868 total time=   0.0s
[CV 5/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.789 total time=   0.0s
[CV 1/5] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.892 total time=   0.0s
[CV 2/5] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.91

In [ ]:
accuracy = accuracy_score(y_test, y_pred_single_PCA)
print(f'Accuracy: {accuracy * 100:.2f}%')

confusion_matrix(y_test.values.ravel(), y_pred_single_PCA)

#### Deseq2

In [8]:
df_train_D2 = pd.concat([X_expression_D2_train, X_clinical_train], axis=1)
df_test_D2 = pd.concat([X_expression_D2_test, X_clinical_test], axis=1)

In [9]:
X_train_D2 = scaler.fit_transform(df_train_D2)
X_test_D2 = scaler.transform(df_test_D2)

In [ ]:
cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'n_estimators': [50, 100, 150],  
    'max_depth': [None, 10, 20], 
    'min_samples_split': [2, 5, 10],  
    'min_samples_leaf': [1, 2, 4], 
    'max_features': ['sqrt', 'log2']
}

classifier = RandomForestClassifier(class_weight="balanced")

Forest_grid_D2 = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=param_grid,   
    n_iter=20,                        
    cv=cross_val,
    n_jobs=1,                         
    scoring='accuracy',
    random_state=42, 
    verbose = 3
)

Forest_grid_D2.fit(X_train_D2, y_train.values.ravel())
y_pred_single_D2 = Forest_grid_D2.predict(X_test_D2)
accuracy = accuracy_score(y_test, y_pred_single_D2)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(confusion_matrix(y_test.values.ravel(), y_pred_single_D2))
pd.DataFrame(confusion_matrix(y_test.values.ravel(), y_pred_single_D2)).to_excel(os.path.join(os.getenv("Output_path"), 'confusion_matrix_RandomForest_D2.xlsx'), sheet_name= 'sheet1', header = True)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.712 total time=   0.0s
[CV 2/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.712 total time=   0.0s
[CV 3/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.712 total time=   0.0s
[CV 4/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.731 total time=   0.0s
[CV 5/5] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.731 total time=   0.0s
[CV 1/5] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.712 total time=   0.1s
[CV 2/5] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.69

### __Logistic Regression__

### PCA

In [ ]:
cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = [
    {'l1_ratio': [0], 'C': np.logspace(-4, 4, 20), 'solver': ['lbfgs'], 'max_iter': [1000, 5000]},
    {'l1_ratio': [0, 1], 'C': np.logspace(-4, 4, 20), 'solver': ['liblinear', 'saga'], 'max_iter': [1000, 5000, 10000]}
]

classifier = LogisticRegression()

LR_grid_PCA = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=param_grid,   
    n_iter=20,                        
    cv=cross_val,
    n_jobs=1,                         
    scoring='accuracy',
    random_state=42, 
    verbose = 3
)

LR_grid_PCA.fit(X_train_PCA, y_train.values.ravel())
y_pred_single_PCA = LR_grid_PCA.predict(X_test_PCA)
accuracy = accuracy_score(y_test, y_pred_single_PCA)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(confusion_matrix(y_test.values.ravel(), y_pred_single_PCA))
pd.DataFrame(confusion_matrix(y_test.values.ravel(), y_pred_single_PCA)).to_excel('confusion_matrix_LR_PCA.xlsx', sheet_name= 'sheet1', header = True)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.750 total time=   0.0s
[CV 2/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.769 total time=   0.0s
[CV 3/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.692 total time=   0.0s
[CV 4/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.692 total time=   0.0s
[CV 5/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.673 total time=   0.0s
[CV 1/5] END C=0.012742749857031334, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.538 total time=   0.0s
[CV 2/5] END C=0.012742749857031334, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.654 total time=   0.0s
[CV 3/5] END C=0.012742749857031334, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.519 total time=   0.0s
[CV 4/5] END C=0.012742749857031334, l1_ratio=1, max_iter=500

c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 2/5] END C=29.763514416313132, l1_ratio=1, max_iter=1000, solver=saga;, score=0.769 total time=   0.0s
[CV 3/5] END C=29.763514416313132, l1_ratio=1, max_iter=1000, solver=saga;, score=0.673 total time=   0.0s
[CV 4/5] END C=29.763514416313132, l1_ratio=1, max_iter=1000, solver=saga;, score=0.654 total time=   0.0s
[CV 5/5] END C=29.763514416313132, l1_ratio=1, max_iter=1000, solver=saga;, score=0.615 total time=   0.0s
[CV 1/5] END C=0.23357214690901212, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.769 total time=   0.0s
[CV 2/5] END C=0.23357214690901212, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.788 total time=   0.0s
[CV 3/5] END C=0.23357214690901212, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.692 total time=   0.0s
[CV 4/5] END C=0.23357214690901212, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.750 total time=   0.0s
[CV 5/5] END C=0.23357214690901212, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.731 total time=   0.0s


c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 1/5] END C=0.615848211066026, l1_ratio=1, max_iter=1000, solver=saga;, score=0.673 total time=   0.0s
[CV 2/5] END C=0.615848211066026, l1_ratio=1, max_iter=1000, solver=saga;, score=0.750 total time=   0.0s
[CV 3/5] END C=0.615848211066026, l1_ratio=1, max_iter=1000, solver=saga;, score=0.654 total time=   0.0s
[CV 4/5] END C=0.615848211066026, l1_ratio=1, max_iter=1000, solver=saga;, score=0.654 total time=   0.0s


...

c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the 

[CV 5/5] END C=206.913808111479, l1_ratio=0, max_iter=1000, solver=saga;, score=0.615 total time=   0.0s
[CV 1/5] END C=0.004832930238571752, l1_ratio=0, max_iter=10000, solver=liblinear;, score=0.596 total time=   0.0s
[CV 2/5] END C=0.004832930238571752, l1_ratio=0, max_iter=10000, solver=liblinear;, score=0.712 total time=   0.0s
[CV 3/5] END C=0.004832930238571752, l1_ratio=0, max_iter=10000, solver=liblinear;, score=0.615 total time=   0.0s
[CV 4/5] END C=0.004832930238571752, l1_ratio=0, max_iter=10000, solver=liblinear;, score=0.577 total time=   0.0s
[CV 5/5] END C=0.004832930238571752, l1_ratio=0, max_iter=10000, solver=liblinear;, score=0.558 total time=   0.0s
[CV 1/5] END C=206.913808111479, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.750 total time=   0.0s
[CV 2/5] END C=206.913808111479, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.769 total time=   0.0s
[CV 3/5] END C=206.913808111479, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.692 total tim

c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 5/5] END C=78.47599703514607, l1_ratio=0, max_iter=1000, solver=saga;, score=0.615 total time=   0.0s
[CV 1/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.538 total time=   0.0s
[CV 2/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.635 total time=   0.0s
[CV 3/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.615 total time=   0.0s
[CV 4/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.538 total time=   0.0s
[CV 5/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.615 total time=   0.0s
[CV 1/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.750 total time=   0.0s
[CV 2/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.769 total time=   0.0s
[CV 3/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.692 total time=   0.0s
[CV 4/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, 

c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Deseq2

In [ ]:
cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = [
    {'l1_ratio': [0], 'C': np.logspace(-4, 4, 20), 'solver': ['lbfgs'], 'max_iter': [1000, 5000]},
    {'l1_ratio': [0, 1], 'C': np.logspace(-4, 4, 20), 'solver': ['liblinear', 'saga'], 'max_iter': [1000, 5000, 10000]}
]

classifier = LogisticRegression()

LR_grid_D2 = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=param_grid,   
    n_iter=20,                        
    cv=cross_val,
    n_jobs=1,                         
    scoring='accuracy',
    random_state=42, 
    verbose = 3
)

LR_grid_D2.fit(X_train_D2, y_train.values.ravel())
y_pred_single_D2 = LR_grid_D2.predict(X_test_D2)
accuracy = accuracy_score(y_test, y_pred_single_D2)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(confusion_matrix(y_test.values.ravel(), y_pred_single_D2))
pd.DataFrame(confusion_matrix(y_test.values.ravel(), y_pred_single_D2)).to_excel(os.path.join(os.getenv("Output_path"), 'confusion_matrix_LR_D2.xlsx'), sheet_name= 'sheet1', header = True)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.577 total time=   0.0s
[CV 2/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.635 total time=   0.0s
[CV 3/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.558 total time=   0.0s
[CV 4/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.481 total time=   0.0s
[CV 5/5] END C=545.5594781168514, l1_ratio=0, max_iter=5000, solver=lbfgs;, score=0.558 total time=   0.0s
[CV 1/5] END C=0.012742749857031334, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.288 total time=   0.0s
[CV 2/5] END C=0.012742749857031334, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.288 total time=   0.0s
[CV 3/5] END C=0.012742749857031334, l1_ratio=1, max_iter=5000, solver=liblinear;, score=0.288 total time=   0.0s
[CV 4/5] END C=0.012742749857031334, l1_ratio=1, max_iter=500

c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 1/5] END C=29.763514416313132, l1_ratio=1, max_iter=1000, solver=saga;, score=0.577 total time=   1.7s


c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 2/5] END C=29.763514416313132, l1_ratio=1, max_iter=1000, solver=saga;, score=0.596 total time=   1.7s


...

[CV 3/5] END C=78.47599703514607, l1_ratio=0, max_iter=1000, solver=saga;, score=0.538 total time=   1.1s


c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 4/5] END C=78.47599703514607, l1_ratio=0, max_iter=1000, solver=saga;, score=0.442 total time=   1.1s


c:\Users\ernes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 5/5] END C=78.47599703514607, l1_ratio=0, max_iter=1000, solver=saga;, score=0.538 total time=   1.3s
[CV 1/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.538 total time=   0.0s
[CV 2/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.558 total time=   0.0s
[CV 3/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.635 total time=   0.0s
[CV 4/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.538 total time=   0.0s
[CV 5/5] END C=0.0001, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.558 total time=   0.0s
[CV 1/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.558 total time=   0.0s
[CV 2/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.558 total time=   0.0s
[CV 3/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, solver=liblinear;, score=0.519 total time=   0.0s
[CV 4/5] END C=78.47599703514607, l1_ratio=0, max_iter=5000, 